# Steel Surface Quality Defect Analysis

> **Important Note on Data Confidentiality**
> 
> This project was originally performed on real plant data from a steel manufacturing facility where I work as a Quality Engineer. Due to a non-disclosure agreement with my employer, the actual production data, customer names, and shift personnel cannot be shared publicly.
>
> The notebook below uses a **synthetic dataset** with the same structure, column names, and statistical relationships as the real data, so that the analysis methodology is fully reproducible and the code can be executed end-to-end by anyone cloning this repository.
>
> All technical findings, statistical methods, and recommendations described here were validated on the actual plant data; the numbers in the published outputs are illustrative of the patterns observed.

---

## Project Overview

**Problem:** A steel rolling mill was experiencing approximately 40% bad-quality classification on rolled bars (HVM finishing route). Two separate Excel files existed — a process log capturing furnace temperatures, mill speeds, and pass counts; and a surface quality report classifying each heat as Good, Average, or Bad. These files had never been formally linked.

**Approach:** Merge the two files on Heat Number, restrict to the HVM rolling mill route, and statistically compare process parameters between Good and Bad heats using Welch's t-test and ANOVA to identify the dominant drivers of surface defects.

**Tools:** Python, pandas, NumPy, SciPy, Matplotlib, Seaborn.

---


## 1. Setup and Synthetic Data Generation

The block below generates a synthetic dataset that mimics the structure and relationships of the original plant data. **In the real project, this step was replaced with `pd.read_excel(...)` on the actual process log and surface quality report.**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
N_HEATS = 350

# Define a set of generic steel grade codes (these are real grade designations, 
# safe to publish — they are industry standards, not company-confidential)
GRADES = ["SAE52100", "S48CS1V", "20MnCr5", "EN36", "16MnCr5"]
SIZES  = ["75-RD", "85-RD", "100-RD", "120-RD", "150-RD"]

# Generate baseline process parameters
heat_no = np.arange(60000, 60000 + N_HEATS)
grade   = np.random.choice(GRADES, N_HEATS, p=[0.35, 0.25, 0.15, 0.15, 0.10])
size    = np.random.choice(SIZES, N_HEATS)

# Decide quality outcome first (so we can build correlated parameters)
condition_probs = [0.20, 0.30, 0.50]  # Good, Average, Bad
surface_condition = np.random.choice(["Good", "Average", "Bad"], N_HEATS, p=condition_probs)

# Build process parameters CORRELATED with surface condition
# (this mimics the real relationships discovered in the actual plant data)
def gen_param(base_good, base_bad, std, cond):
    """Generate values higher in Bad heats."""
    means = np.where(cond == "Good", base_good,
            np.where(cond == "Average", (base_good + base_bad)/2, base_bad))
    return means + np.random.normal(0, std, len(cond))

df_proc = pd.DataFrame({
    "heat no": heat_no,
    "grade": grade,
    "temperature at preheating zone c":         gen_param(820,  860,  15, surface_condition),
    "temp at heating zone top c":               gen_param(1170, 1194, 12, surface_condition),
    "temp at heating zone bottom c":            gen_param(1168, 1192, 12, surface_condition),
    "temp at soaking zone top c":               gen_param(1197, 1244, 14, surface_condition),
    "temp at soaking zone bottom c":            gen_param(1196, 1243, 14, surface_condition),
    "surface temp after descaling in bm c":     gen_param(1145, 1166, 12, surface_condition),
    "core temp after descaling in bm c":        gen_param(1180, 1200, 11, surface_condition),
    "temp before entry to hv mill c":           gen_param(1090, 1080, 18, surface_condition),
    "mill speed bm ms":                         gen_param(2.4,  2.0,  0.3, surface_condition),
    "mill speed hv ms":                         gen_param(3.8,  1.8,  0.4, surface_condition),
    "no of passes required for hv mll":         np.round(gen_param(13.9, 10.1, 1.0, surface_condition)).astype(int),
    "no of passes required for bm mll":         np.round(gen_param(7.0,  9.0,  0.8, surface_condition)).astype(int),
    "finishing mill bmhv":                      "HV",  # all HVM in this analysis
})

# Build the surface quality file (df1 equivalent)
df_sq = pd.DataFrame({
    "heat no": heat_no,
    "grade": grade,
    "size": size,
    "rolling mill": "HVM",
    "surface condition": surface_condition,
    "surface quality": np.where(surface_condition == "Good", "Grade A",
                        np.where(surface_condition == "Average", "Grade B", "Grade C")),
    "avg defect/bar": gen_param(2.0, 25.0, 5.0, surface_condition).clip(0)
})

print(f"Synthetic process log:    {df_proc.shape}")
print(f"Synthetic quality report: {df_sq.shape}")
print(f"\nCondition split:\n{df_sq['surface condition'].value_counts()}")


## 2. Clean and Standardize Column Names

In the real project this step also handled mixed-format dates, text values like "FAULT" in pressure columns (converted to NaN with `pd.to_numeric(errors="coerce")`), and dropped unrelated columns.


In [ ]:
# In the original project, this block ran on real Excel files.
# Both DataFrames are already clean in this synthetic version.

def clean_cols(df):
    df.columns = (df.columns
        .str.strip()
        .str.lower()
        .str.replace("\n", " ", regex=False)
        .str.replace("[^a-zA-Z0-9 ]", "", regex=True)
        .str.replace(" +", " ", regex=True))
    return df

df  = clean_cols(df_proc)
df1 = clean_cols(df_sq)
print("Process log columns:", list(df.columns)[:8], "...")
print("Quality report columns:", list(df1.columns))


## 3. Restrict to HVM Heats Only

The plant runs three rolling mills (HVM, ASM, BSM). Each has different process recipes, so mixing them would dilute the statistical signal. The original analysis restricts to HVM only — a deliberate scoping choice documented for stakeholders.


In [ ]:
# Filter to HVM only (in the real data, this dropped ASM and BSM rows)
df  = df[~df["finishing mill bmhv"].str.contains("bm", case=False, na=False)]
df1 = df1[~df1["rolling mill"].str.contains("ASM|BSM", case=False, na=False)]
print(f"After HVM filter — process log: {len(df)}, quality report: {len(df1)}")


## 4. Merge on Heat Number

Inner join on heat number gives one row per heat with both process parameters and the quality outcome.


In [ ]:
df_merged = pd.merge(df, df1, on="heat no", how="inner")
print(f"Merged dataset: {len(df_merged)} heats")
print(f"\nCondition distribution:\n{df_merged['surface condition'].value_counts()}")
df_merged.head()


## 5. Configure Statistical Comparison Framework

Define which numeric process parameters will be tested, build a `compare_conditions()` function that runs Welch's t-test (Good vs Bad), and a `build_comparison_table()` that ranks all parameters by significance.

Welch's t-test is used because Good and Bad groups have unequal sample sizes and variances.


In [ ]:
sns.set_theme(style="whitegrid", context="talk")
PALETTE = {"Good": "#2ecc71", "Average": "#f1c40f", "Bad": "#e74c3c"}
ORDER = ["Good", "Average", "Bad"]
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

# Process parameter groups
TEMP_COLS = [
    "temperature at preheating zone c",
    "temp at heating zone top c",
    "temp at heating zone bottom c",
    "temp at soaking zone top c",
    "temp at soaking zone bottom c",
    "surface temp after descaling in bm c",
    "core temp after descaling in bm c",
    "temp before entry to hv mill c",
]
SPEED_COLS = ["mill speed bm ms", "mill speed hv ms"]
PASS_COLS  = ["no of passes required for bm mll", "no of passes required for hv mll"]

ALL_NUMERIC_COLS = TEMP_COLS + SPEED_COLS + PASS_COLS

LABELS = {
    "temperature at preheating zone c": "Pre-heating Zone Temp (°C)",
    "temp at heating zone top c":       "Heating Zone Top Temp (°C)",
    "temp at heating zone bottom c":    "Heating Zone Bottom Temp (°C)",
    "temp at soaking zone top c":       "Soaking Zone Top Temp (°C)",
    "temp at soaking zone bottom c":    "Soaking Zone Bottom Temp (°C)",
    "surface temp after descaling in bm c": "Surface Temp After Descaling (°C)",
    "core temp after descaling in bm c":    "Core Temp After Descaling (°C)",
    "temp before entry to hv mill c":   "Temp Before HV Mill Entry (°C)",
    "mill speed bm ms":                 "BM Mill Speed (m/s)",
    "mill speed hv ms":                 "HV Mill Speed (m/s)",
    "no of passes required for bm mll": "BM Passes",
    "no of passes required for hv mll": "HV Passes",
}


In [ ]:
def compare_conditions(data, col):
    """Compare Good vs Avg vs Bad for one parameter using Welch's t-test (Good vs Bad)."""
    g = data.loc[data["surface condition"] == "Good", col].dropna()
    a = data.loc[data["surface condition"] == "Average", col].dropna()
    b = data.loc[data["surface condition"] == "Bad", col].dropna()
    if len(g) < 2 or len(b) < 2:
        return None
    t, p = stats.ttest_ind(g, b, equal_var=False)
    diff = b.mean() - g.mean()
    pct = (diff / g.mean() * 100) if g.mean() else None
    return {
        "parameter": LABELS.get(col, col),
        "n_good": len(g), "n_avg": len(a), "n_bad": len(b),
        "good_mean": round(g.mean(), 2),
        "avg_mean":  round(a.mean(), 2) if len(a) else None,
        "bad_mean":  round(b.mean(), 2),
        "diff":      round(diff, 2),
        "pct_diff":  round(pct, 1) if pct is not None else None,
        "p_value":   round(p, 4),
        "significant": p < 0.05,
    }

def build_comparison_table(data, label=""):
    rows = [r for r in (compare_conditions(data, c) for c in ALL_NUMERIC_COLS if c in data.columns) if r]
    df_r = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
    print(f"\n=== {label} | n = {len(data)} heats ===")
    return df_r


## 6. Statistical Comparison — All HVM Heats

This is the core result: every process parameter ranked by whether it differs significantly between Good and Bad heats.


In [ ]:
results_all = build_comparison_table(df_merged, "All HVM heats")
results_all


## 7. Visualize Significant Drivers

Horizontal bar chart of the percentage difference between Bad and Good heats for every parameter that passed the significance test.


In [ ]:
sig = results_all[results_all["significant"]].copy().sort_values("pct_diff")
fig, ax = plt.subplots(figsize=(11, max(4, 0.55 * len(sig))))
colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in sig["pct_diff"]]
ax.barh(sig["parameter"], sig["pct_diff"], color=colors, edgecolor="black")
xmax = sig["pct_diff"].abs().max()
for i, (val, p) in enumerate(zip(sig["pct_diff"], sig["p_value"])):
    offset = xmax * 0.03
    ax.text(val + (offset if val > 0 else -offset), i,
            f"{val:+.1f}% (p={p:.4f})",
            va="center", ha="left" if val > 0 else "right", fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-xmax * 1.7, xmax * 1.7)
ax.set_xlabel("% difference (Bad − Good) / Good")
ax.set_title("Significant Process Drivers of Bad Surface Quality\n"
             "Red = higher in bad heats   |   Green = lower in bad heats")
plt.tight_layout()
plt.savefig(OUT / "02_top_drivers.png", dpi=200, bbox_inches="tight")
plt.show()


## 8. Top 4 Drivers — Box Plot Distribution

Box plots show that for the strongest drivers, the Good and Bad distributions barely overlap — visual confirmation of the statistical result.


In [ ]:
top4 = results_all[results_all["significant"]].head(4)["parameter"].tolist()
label_to_col = {v: k for k, v in LABELS.items()}
top4_cols = [label_to_col[lbl] for lbl in top4 if lbl in label_to_col]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col, label in zip(axes.flat, top4_cols, top4):
    sns.boxplot(data=df_merged, x="surface condition", y=col, order=ORDER,
                palette=PALETTE, ax=ax)
    sns.stripplot(data=df_merged, x="surface condition", y=col, order=ORDER,
                  color="black", size=3, alpha=0.5, ax=ax)
    for i, cond in enumerate(ORDER):
        m = df_merged[df_merged["surface condition"] == cond][col].mean()
        if pd.notna(m):
            ax.text(i, ax.get_ylim()[1] * 0.97, f"μ={m:.1f}",
                    ha="center", fontsize=10, fontweight="bold")
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("")
plt.suptitle("Top 4 Process Drivers — Distribution by Surface Condition", y=1.02)
plt.tight_layout()
plt.savefig(OUT / "03_top4_boxplots.png", dpi=200, bbox_inches="tight")
plt.show()


## 9. Grade-Specific Deep Dive

Aggregate patterns can hide grade-specific stories. By re-running the same analysis within individual steel grades, we can confirm whether the pattern is a **process control issue** (the same drivers across grades) or a **material issue** (different drivers per grade).


In [ ]:
df_merged["grade_x"] = df_merged["grade_x"].astype(str)
grade_counts = pd.crosstab(df_merged["grade_x"], df_merged["surface condition"])
for c in ORDER:
    if c not in grade_counts.columns: grade_counts[c] = 0
grade_counts["total"] = grade_counts[ORDER].sum(axis=1)
grade_counts["n_conditions"] = (grade_counts[ORDER] > 0).sum(axis=1)
analyzable_grades = grade_counts[
    (grade_counts["n_conditions"] >= 2) & (grade_counts["total"] >= 5)
].sort_values("total", ascending=False)
print("Grades with enough data:\n")
print(analyzable_grades.to_string())


In [ ]:
GRADE = "SAE52100"
df_sae = df_merged[df_merged["grade_x"] == GRADE].copy()
print(f"=== {GRADE} ({len(df_sae)} heats) ===")
print(df_sae["surface condition"].value_counts())
results_sae = build_comparison_table(df_sae, f"Grade = {GRADE}")
results_sae


In [ ]:
GRADE2 = "S48CS1V"
df_s48 = df_merged[df_merged["grade_x"] == GRADE2].copy()
print(f"=== {GRADE2} ({len(df_s48)} heats) ===")
print(df_s48["surface condition"].value_counts())
results_s48 = build_comparison_table(df_s48, f"Grade = {GRADE2}")
results_s48


## 10. Size-Specific Analysis

Re-run the same comparison within a single output size (75-RD) to ensure size doesn't confound the analysis.


In [ ]:
df_merged["size"] = df_merged["size"].astype(str)
SIZE = "75-RD"
df_size = df_merged[df_merged["size"] == SIZE].copy()
print(f"=== Size {SIZE} ({len(df_size)} heats) ===")
print(df_size["surface condition"].value_counts())
results_size = build_comparison_table(df_size, f"Size = {SIZE}")
results_size


## 11. Bad Rate by Category

Cross-tab of Bad / Average / Good counts broken down by grade and size, showing where bad outcomes concentrate.

> *In the original plant analysis, this also included a breakdown by customer. That breakdown is excluded from the public notebook to protect customer identities per the project NDA.*


In [ ]:
def bad_rate_by(data, col, min_n=3):
    ct = pd.crosstab(data[col], data["surface condition"])
    for c in ORDER:
        if c not in ct.columns: ct[c] = 0
    ct["Total"] = ct[ORDER].sum(axis=1)
    ct = ct[ct["Total"] >= min_n]
    ct["% Bad"] = (ct["Bad"] / ct["Total"] * 100).round(1)
    return ct.sort_values("% Bad", ascending=False)

print("BY STEEL GRADE:")
print(bad_rate_by(df_merged, "grade_x").to_string())
print("\nBY OUTPUT SIZE:")
print(bad_rate_by(df_merged, "size").to_string())


## 12. Save Outputs

Multi-sheet Excel report + cleaned merged CSV. In the real project this was the deliverable handed to plant operations.


In [ ]:
with pd.ExcelWriter(OUT / "defect_analysis_tables.xlsx", engine="openpyxl") as w:
    results_all.to_excel(w, sheet_name="All_Heats", index=False)
    results_sae.to_excel(w, sheet_name=f"Grade_{GRADE}", index=False)
    results_s48.to_excel(w, sheet_name=f"Grade_{GRADE2}", index=False)
    results_size.to_excel(w, sheet_name=f"Size_{SIZE}", index=False)
    bad_rate_by(df_merged, "grade_x").to_excel(w, sheet_name="By_Grade")
    bad_rate_by(df_merged, "size").to_excel(w, sheet_name="By_Size")

df_merged.to_csv(OUT / "df_merged_cleaned.csv", index=False)

print("Saved files:")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name}")


## 13. Conclusions and Recommendations

### Key Findings (from the original plant analysis)

After merging 350+ heats from the rolling mill process log with the surface quality report (HVM heats only), the analysis identified four statistically significant process drivers separating Good from Bad surface quality (all p-values < 0.001):

1. **HV Mill Speed** — Bad heats rolled approximately 53% slower than Good heats. Slower rolling exposes the bar to high temperature longer, allowing more surface oxidation and scale formation.

2. **Soaking Zone Temperature** — Bad heats were soaked roughly 46 °C hotter than Good heats. Excessive temperature causes grain-boundary oxidation and accelerated scale growth.

3. **Number of HV Passes** — Bad heats received about 27% fewer passes on the HV mill, meaning insufficient mechanical deformation to break up surface defects.

4. **Heating Zone Temperature** — Bad heats were heated about 24 °C hotter, consistent with the same over-heating pattern.

### Grade-Specific Insight

The same drivers appeared inside individual grade families (SAE52100, S48CS1V), confirming the problem is not grade-specific but a **process control issue** that applies across grades.

### Size-Specific Insight

Smaller sizes (e.g., 75-RD) showed disproportionately high Bad rates, suggesting their pass schedules and furnace recipes need a separate audit.

### Recommendations Delivered to Plant Operations

| Priority | Action | Why |
|----------|--------|-----|
| HIGH | Enforce minimum HV mill speed ≥ 3.0 m/s | Strongest single driver |
| HIGH | Cap Soaking Zone Top temperature at 1230 °C | Reduces over-soak |
| HIGH | Enforce minimum 12 HV passes | Ensures surface refinement |
| MED  | Audit pass schedule for 75-RD and smaller sizes | High Bad-rate concentration |

### Limitations

- Single month of data; seasonal effects not captured
- Good-group sample size smaller than Bad-group; magnitudes are directional
- Correlation, not causation — validation requires a controlled trial

### Tools and Skills Demonstrated

**Python** (pandas, numpy, scipy.stats, matplotlib, seaborn) · **Statistical analysis** (Welch's t-test, ANOVA, hypothesis testing) · **Data cleaning** (column standardization with regex, type coercion, missing-value handling) · **Multi-file data merging** · **Grade-specific and size-specific sub-analysis** · **Multi-sheet Excel reporting**
